# Apparent magnitude estimation

Scratchpad for experimenting with turning the `source`/`image`/`flux` tables produced by
`pw-aperture` into calibrated apparent magnitudes, ahead of implementing
`photometry_workflow.apparent_magnitude.api.estimate_apparent_magnitude`.

Approach used below (method 1 from `src/photometry_workflow/apparent_magnitude/CLAUDE.md`):
for each image, take the sources with a Gaia match (`gaia_matched`) as calibrators, compute a
single zero point from them, and apply it to every source's instrumental flux in that image.

Still to try, per the module's notes:
- weight calibrator contributions to the zero point (e.g. by flux scatter across apertures/images)
- iterative minimization of check-star magnitude variance instead of a plain median zero point
- selecting comparison stars from Gaia color/magnitude rather than using every matched source
- propagating flux/background uncertainty through to a magnitude error

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, join

## Load the tables

Point `reduce_dir` at a folder written by `pw-aperture` (i.e. containing `sources.*`,
`images.*`, `flux.*`).

In [ ]:
reduce_dir = Path("../reduce")

source_table = Table.read(next(reduce_dir.glob("sources.*")))
image_table = Table.read(next(reduce_dir.glob("images.*")))
flux_table = Table.read(next(reduce_dir.glob("flux.*")))

len(source_table), len(image_table), len(flux_table)

In [ ]:
source_table

In [ ]:
image_table

## Pick an aperture

`flux_table["flux"]` and `flux_table["bkg"]` are vectors over the relative aperture radii
used by `pw-aperture` (`relative_apertures_radii` in `aperture_photometry/api.py`). Start with a
single fixed aperture index picked by eye; revisit with a proper curve-of-growth choice later.

In [ ]:
aperture_index = 20

flux_table["flux_1ap"] = flux_table["flux"][:, aperture_index]
flux_table["bkg_1ap"] = flux_table["bkg"][:, aperture_index]

## Per-image zero point from Gaia-matched calibrators

In [ ]:
calibrators = source_table[source_table["gaia_matched"]]["source_id", "phot_g_mean_mag"]

zero_points = []
for image_id in image_table["image_id"]:
    image_flux = flux_table[flux_table["image_id"] == image_id]
    calibrator_flux = join(image_flux, calibrators, keys="source_id")

    instrumental_mag = -2.5 * np.log10(calibrator_flux["flux_1ap"])
    zero_point = np.median(calibrator_flux["phot_g_mean_mag"] - instrumental_mag)
    zero_points.append(zero_point)

image_table["zero_point"] = zero_points
image_table["image_id", "time", "zero_point"]

## Apply the zero point to every source

In [ ]:
flux_table = join(flux_table, image_table["image_id", "time", "zero_point"], keys="image_id")
instrumental_mag = -2.5 * np.log10(flux_table["flux_1ap"])
flux_table["magnitude"] = instrumental_mag + flux_table["zero_point"]

## Light curve for a single source

Set `target_source_id` to the `source_id` of the star of interest (look it up in
`source_table` by `centroid_x`/`centroid_y` or the cross-matched Gaia columns).

In [ ]:
target_source_id = 0

target_light_curve = flux_table[flux_table["source_id"] == target_source_id]
target_light_curve.sort("time")

plt.figure(figsize=(8, 4))
plt.plot(target_light_curve["time"], target_light_curve["magnitude"], ".")
plt.gca().invert_yaxis()
plt.xlabel("time (JD)")
plt.ylabel("apparent magnitude")
plt.title(f"source_id={target_source_id} apparent magnitude")
plt.show()